In [ ]:
from functools import partial
import numpy as np
import jax
import jax.numpy as jnp
from jax import vmap

from geospsr.kernels import Matern32Kernel, ProductKernel
from geospsr.utils import trunc_Zd, interpolator
from geospsr.gp import (poisson_cross_covariances, compute_mean,
                        sample_pathwise_conditioning,
                        f_eigenvalues_from_v_eigenvalues, f_gamma_from_v_xi)
from jax.scipy.interpolate import RegularGridInterpolator

from skimage.measure import marching_cubes
from scipy.spatial import KDTree
import open3d as o3d
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyprojroot import here
from pathlib import Path
from huggingface_hub import snapshot_download

In [ ]:
DATA_DIR = here("data/ppsurf")

if not (DATA_DIR / "famous_noisefree").exists():
    snapshot_download(
        "perler/ppsurf",
        repo_type="dataset",
        allow_patterns=["famous_*/*"],
        local_dir=DATA_DIR
    )

In [ ]:
VARIANTS = ["famous_noisefree", "famous_original", "famous_sparse",
            "famous_dense", "famous_extra_noisy"]
SHAPES = ["xyzrgb_statuette"]           # (was: dragon, bunny, Armadillo)


def load_points(variant: str, shape: str) -> np.ndarray:
    path = DATA_DIR / variant / "04_pts_vis" / f"{shape}.xyz.ply"
    return np.asarray(o3d.io.read_point_cloud(str(path)).points)


def normals_pca(points, k=50) -> np.ndarray:
    '''
    출처는 주석 바로 남긴다
    '''
    
    tree = KDTree(points)
    dist, idx = tree.query(x=points, k=k, workers=-1)

    neighbors = points[idx]
    centered = neighbors - neighbors.mean(axis=1, keepdims=True)

    cov = np.einsum("nki,nkj->nij", centered, centered) / k
    w, V = np.linalg.eigh(cov)
    normals = V[:, :, 0]
    return normals


def normals_o3d(points, k=30, orient_k=20) -> np.ndarray:
    # k / orient_k from a sweep vs GT normals (statuette, all 5 variants):
    # k=30/orient_k=20 orients >= 96% of points correctly except extra_noisy (72%);
    # k=20/orient_k=15 drops dense to 95% and extra_noisy to 55% (~random)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(
        np.asarray(points, dtype=np.float64))
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamKNN(knn=k)
    )
    pcd.orient_normals_consistent_tangent_plane(k=orient_k)

    c = np.mean(points, axis=0)
    n = np.asarray(pcd.normals)
    if np.mean(np.sum((points - c) * n, axis=1)) < 0:
        n = -n
    return n


def normals_gt(points, gt_mesh) -> np.ndarray:
    """
    Normal of the closest GT triangle for every point (both in dataset coordinates):
    the "perfect normals" comparison, which removes the normal-estimation error
    """
    gt_mesh.compute_triangle_normals()
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(gt_mesh))
    hit = scene.compute_closest_points(
        o3d.core.Tensor(np.asarray(points, dtype=np.float32)))
    return np.asarray(gt_mesh.triangle_normals)[hit["primitive_ids"].numpy()]


def normal_segments(points, normals, length=0.03, every=20):
    """
    Line segments p -> p + length * n for every `every`-th point, as x/y/z
    arrays separated by None (one plotly trace draws all segments)
    """
    p = points[::every]
    q = p + length * normals[::every]
    seg = np.full((len(p) * 3, 3), np.nan)
    seg[0::3], seg[1::3] = p, q
    return [np.where(np.isnan(c), None, c) for c in seg.T]


def normalize_points(v):
    """
    Normalize points to tightly fit in [-0.5, 0.5]^d

    v: (n, d)

    returns: (n, d)
    """
    v = v - v.min(0)
    v = v / v.max()
    v = v - 0.5 * v.max(0)
    return v


def visualize_normals(points, normals_dict, length=0.03, every=20,
                      point_size=1.5, height=550):
    """
    Inline (notebook) plot, one 3D subplot per entry, no separate window.
    normals_dict: {"PCA": normals, "o3d": normals, ...}
    Points are colored by normal direction (n + 1) / 2 -> RGB, and every
    `every`-th normal is drawn as a black segment.
    """
    points = normalize_points(points)
    names = list(normals_dict)
    fig = make_subplots(
        rows=1, cols=len(names), subplot_titles=names,
        specs=[[{"type": "scene"}] * len(names)],
        horizontal_spacing=0.01,
    )
    for i, name in enumerate(names, start=1):
        nrm = np.asarray(normals_dict[name])
        rgb = np.clip((nrm + 1) / 2 * 255, 0, 255).astype(int)
        fig.add_trace(go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="markers",
            marker=dict(size=point_size,
                        color=[f"rgb({r},{g},{b})" for r, g, b in rgb]),
            name=f"{name} points", showlegend=False, hoverinfo="skip",
        ), row=1, col=i)
        sx, sy, sz = normal_segments(points, nrm, length, every)
        fig.add_trace(go.Scatter3d(
            x=sx, y=sy, z=sz, mode="lines",
            line=dict(color="black", width=1),
            name=f"{name} normals", showlegend=False, hoverinfo="skip",
        ), row=1, col=i)
    fig.update_scenes(aspectmode="data", xaxis_visible=False,
                      yaxis_visible=False, zaxis_visible=False)
    fig.update_layout(height=height, margin=dict(l=0, r=0, t=30, b=0))
    fig.show()


# Modified cahcable version of the original function to use cached values
def cachable_periodic_stationary_interpolator(
    f, dim, density, bs=128, exponent=1, verbose=False, cache_path=None
):
    """
    Create an interpolating object for a periodic stationary kernel with period [0, 2pi]
    Uses grid values of (linspace(-1, 1, density) ** exponent * jnp.pi)^dim

    cache_path: grid values are loaded from here if it exists, otherwise computed
    and saved here. The values depend on the kernel hyperparameters, so the path
    must encode all of them (and dim, density, exponent).
    """
    xis = (jnp.mgrid[-1: 1: density * 1j] ** exponent) * jnp.pi
    points = (xis,) * dim

    if cache_path is not None and Path(cache_path).exists():
        values = jnp.asarray(np.load(cache_path))
        if values.shape[:dim] != (density,) * dim:
            raise ValueError(
                f"cached grid shape {values.shape} does not match "
                f"density={density}, dim={dim}: {cache_path}"
            )
        f_interpolator = RegularGridInterpolator(points, values)
        if verbose:
            print(f"Loaded interpolator values from {cache_path}")
    else:
        # jit: the library's bmap runs vmap eagerly (much slower, esp. on GPU)
        f_interpolator = interpolator(jax.jit(f), points, bs, verbose)
        if cache_path is not None:
            Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
            np.save(cache_path, np.asarray(f_interpolator.values))
            if verbose:
                print(f"Saved interpolator values to {cache_path}")

    def helper(x1, x2):
        x_diff = x1 - x2
        x_diff = (x_diff + jnp.pi) % (2 * jnp.pi) - jnp.pi
        return f_interpolator(x_diff).squeeze()

    return helper


# 사용
pts = load_points("famous_noisefree", SHAPES[0])
visualize_normals(pts, {"PCA": normals_pca(pts), "o3d": normals_o3d(pts)})





In [ ]:
print("JAX devices:", jax.devices())   # want CudaDevice (WSL2 + jax[cuda12])

# Fixed across configs. NB: SDD uses sigma**2 as the noise variance and the
# product kernel's variance is variance**3 = 1e-3 -> signal/noise ~ 2.5.
# (The Cholesky path in gp.py adds sigma, not sigma**2.)
sigma = 0.02
variance = 0.1
interp_exponent = 5

# Units: Matern32Kernel evaluates at x / (2 pi), so `lengthscale` is in units of
# the period. In the normalized coords (data in [-pi/2, pi/2]) it is 2 pi * ls:
# 0.063 / 0.044 / 0.031 below, vs. mean nearest-neighbour spacing of the statuette
# clouds 0.021 (dense) .. 0.044 (sparse, extra_noisy).
# Detail is set by the lengthscale: the Matern32 spectrum halves at frequency
# ~0.177 / lengthscale. Truncation n must grow ~1/lengthscale so the truncated
# k_fv still holds ~85% of the kernel's spectral mass (k_v in the solve is NOT
# truncated, so a small n makes the two inconsistent). Cost grows as n^3.
#   ls 1e-2 / n 50 : 86%    ls 7e-3 / n 70 : 85%    ls 5e-3 / n 100 : 86%
# grid: 128 for every config, so configs differ only in the lengthscale. (statuette,
# k=50 normals: ls0.007/r128 beat ls0.01/r64 on every variant, CD -6..-17%, but
# that run changed lengthscale and grid together.)
CONFIGS = [
    dict(lengthscale=1e-2, n=50, grid=128, density=50),    # baseline
    dict(lengthscale=7e-3, n=70, grid=128, density=50),
    dict(lengthscale=5e-3, n=100, grid=128, density=50),   # k_fv cache ~5 h on CPU
]
for c in CONFIGS:
    c["name"] = f"ls{c['lengthscale']:g}_n{c['n']}_r{c['grid']}"


def freq_batch_size(n_freq, floats_per_freq, mem_budget=2**30, max_bs=512):
    """points per batch so a (bs, n_freq, floats_per_freq) temporary fits mem_budget"""
    return int(np.clip(mem_budget // (n_freq * floats_per_freq * 4), 1, max_bs))


def kfv_interp_error(kern, n_test=256, seed=0):
    """
    max |interpolated - exact| / max |exact| of k_fv at random offsets
    (half near 0 where the kernel is peaked, half uniform). Checks `density`.
    """
    r = np.random.default_rng(seed)
    d = np.concatenate([r.normal(0, 0.2, (n_test // 2, 3)),
                        r.uniform(-np.pi, np.pi, (n_test // 2, 3))])
    d = jnp.asarray(np.clip(d, -np.pi, np.pi), dtype=jnp.float32)
    # batched: all n_test at once is (n_test, n_freq) -> 8 GB at n=100
    f_exact = jax.jit(vmap(kern["k_fv_exact"]))
    bs = freq_batch_size(len(kern["v_evecs"]), 8)
    exact = jnp.concatenate([f_exact(d[i:i + bs]) for i in range(0, len(d), bs)])
    approx = vmap(kern["k_fv"], (0, None))(d, jnp.zeros(3))
    return float(jnp.abs(approx - exact).max() / jnp.abs(exact).max())


def make_kernels(cfg):
    """Everything that depends on one config, as a dict. k_fv grid values are cached."""
    k1 = Matern32Kernel(cfg["lengthscale"], variance)
    k_v = ProductKernel(k1, k1, k1)            # closed form, used in the solve
    evecs = trunc_Zd(cfg["n"])
    evals = vmap(k_v.spectral_density)(evecs) ** 0.5
    k_fv_exact = partial(poisson_cross_covariances, eigenvectors=evecs, eigenvalues=evals)
    cache_path = here(
        "data/cache/kfv"
        f"_ls{cfg['lengthscale']}_var{variance}_n{cfg['n']}"
        f"_dens{cfg['density']}_exp{interp_exponent}_dim3.npy"
    )
    k_fv = cachable_periodic_stationary_interpolator(
        k_fv_exact, 3, cfg["density"],
        bs=freq_batch_size(len(evecs), 8),     # (bs, n_freq, 3) + sin/dot temporaries
        exponent=interp_exponent, verbose=True, cache_path=cache_path,
    )
    return dict(cfg=cfg, k_v=k_v, k_fv=k_fv, k_fv_exact=k_fv_exact,
                v_evecs=evecs, v_evals=evals,
                f_evals=f_eigenvalues_from_v_eigenvalues(evecs, evals))

In [ ]:
# ---- Batch experiment: every SHAPE x VARIANT -> mean mesh (SDD) -> metrics ----
import csv
import time
from jax import random


def normalization_transform(points):
    """
    Same transform as `normalize_points(points) * pi`, but returned as
    (center, scale) so it can be applied to other geometry (the GT mesh)
    and inverted: x = (p - center) / scale * pi
    """
    lo, hi = points.min(0), points.max(0)
    return (lo + hi) / 2, (hi - lo).max()


def make_grid(res, half_extent):
    """(res^3, 3) grid on [-half_extent, half_extent]^3, 'ij' ordering"""
    lin = np.linspace(-half_extent, half_extent, res)
    g = np.stack(np.meshgrid(lin, lin, lin, indexing="ij"), axis=-1)
    return g.reshape(-1, 3), lin


def query_batch_size(n_data, mem_budget=2**30):
    """compute_mean holds a (bs, n_data, 3) block plus interpolation temporaries"""
    return int(np.clip(mem_budget // (n_data * 3 * 4 * 10), 32, 4096))


def mesh_metrics(rec, gt, n_samples=100_000, fscore_tau=0.01):
    """Chamfer (L1, mean of both directions), normal consistency, F-score@tau"""
    pa = rec.sample_points_uniformly(n_samples, use_triangle_normal=True)
    pb = gt.sample_points_uniformly(n_samples, use_triangle_normal=True)
    a, na = np.asarray(pa.points), np.asarray(pa.normals)
    b, nb = np.asarray(pb.points), np.asarray(pb.normals)

    d_ab, i_ab = KDTree(b).query(a, workers=-1)
    d_ba, i_ba = KDTree(a).query(b, workers=-1)

    chamfer = 0.5 * (d_ab.mean() + d_ba.mean())
    # orientation-free: |cos| between matched normals
    nc = 0.5 * (np.abs((na * nb[i_ab]).sum(1)).mean()
                + np.abs((nb * na[i_ba]).sum(1)).mean())
    precision = (d_ab < fscore_tau).mean()
    recall = (d_ba < fscore_tau).mean()
    fscore = 2 * precision * recall / max(precision + recall, 1e-12)
    return chamfer, nc, fscore


def reconstruct(points, kern, sdd_params, normals=None, pad=1.1):
    """
    points: raw (n, 3) cloud in dataset coordinates
    kern: dict from `make_kernels` (its cfg sets the marching cubes grid)
    normals: (n, 3) oriented normals of `points`; None -> estimated (`normals_o3d`)
    returns: o3d mesh in dataset coordinates, timings dict
    """
    grid_res = kern["cfg"]["grid"]
    t = {}
    center, scale = normalization_transform(points)
    x = (points - center) / scale * np.pi          # data in [-pi/2, pi/2]

    t0 = time.perf_counter()
    # uniform scaling keeps directions, so given normals are valid for x as well
    n = normals_o3d(x) if normals is None else np.asarray(normals)
    t["normals"] = time.perf_counter() - t0

    half = np.pi / 2 * pad                        # < pi, so no periodic wrap
    x_grid, lin = make_grid(grid_res, half)

    x_data = jnp.asarray(x)
    v_data = jnp.asarray(n)
    # one SDD solve for data points + grid together (compute_mean re-solves per call)
    x_query = jnp.concatenate([x_data, jnp.asarray(x_grid)])

    t0 = time.perf_counter()
    mean = compute_mean(
        x_query, x_data, v_data, kern["k_v"], kern["k_fv"], sigma,
        bs=query_batch_size(len(x)), sdd_params=sdd_params,
    )
    # np.array (copy): np.asarray of a jax array is read-only -> marching_cubes fails
    mean = np.array(mean.block_until_ready())
    t["solve+eval"] = time.perf_counter() - t0

    data_mean, grid_mean = mean[: len(x)], mean[len(x):]
    iso = data_mean.mean()
    vol = grid_mean.reshape(grid_res, grid_res, grid_res)
    if not vol.min() < iso < vol.max():
        raise RuntimeError(
            f"iso {iso:.4g} outside volume range [{vol.min():.4g}, {vol.max():.4g}]")

    t0 = time.perf_counter()
    step = lin[1] - lin[0]
    verts, faces, _, _ = marching_cubes(vol, level=iso, spacing=(step,) * 3)
    verts = verts + lin[0]                         # index -> normalized coords
    verts = verts / np.pi * scale + center         # normalized -> dataset coords
    t["marching_cubes"] = time.perf_counter() - t0

    mesh = o3d.geometry.TriangleMesh(
        o3d.utility.Vector3dVector(verts), o3d.utility.Vector3iVector(faces))
    return mesh, t

In [ ]:
# SDD solve. The stochastic gradient is scaled by N/bs, so the largest stable step
# shrinks ~1/N (statuette, ls 0.01: lr=1 converges at N=8.5k but diverges at N=24k).
# lr = SDD_LR_TIMES_N / N for a fixed number of epochs reached residual < 1e-3 on
# all 5 variants (ls 0.01 and 0.005). The old fixed lr=1e-3 / 1000 iterations
# stopped at ~90% residual: an under-fitted, blurred mean surface (noisefree,
ls 0.01, r64: CD 0.0085 -> 0.0061, F1 0.67 -> 0.84 once converged).
SDD_LR_TIMES_N = 6000
SDD_EPOCHS = 40


def sdd_params_for(n_points, key=random.key(0), bs=128):
    return {"key": key, "bs": bs, "lr": SDD_LR_TIMES_N / n_points,
            "iterations": int(np.ceil(SDD_EPOCHS * n_points / bs))}


RUN_VARIANTS = ["famous_noisefree", "famous_original", "famous_extra_noisy",
                "famous_sparse", "famous_dense"]
RUN_SHAPES = SHAPES
RUN_CONFIGS = CONFIGS            # e.g. CONFIGS[:2] to skip the n=100 one
# meshes + results.csv of the config comparison (the demo adds figures/ here)
OUT_DIR = here("data/results/geospsr_configs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


# "gt": normals of the closest GT triangle instead of estimated ones, i.e. the same
# method without the normal-estimation error (doubles the run time; ("est",) to skip)
RUN_NORMALS = ("est", "gt")
NORMALS_SUFFIX = {"est": "", "gt": "_gtnormals"}


def result_stem(shape, variant, label):
    """label: config name + NORMALS_SUFFIX"""
    return f"{shape}_{variant}_{label}"


results = []
for cfg in RUN_CONFIGS:
    t0 = time.perf_counter()
    kern = make_kernels(cfg)                   # first time per config: builds k_fv cache
    t_kernel = time.perf_counter() - t0
    interp_err = kfv_interp_error(kern)
    print(f"== {cfg['name']}: kernels {t_kernel:.0f}s, k_fv interp rel. error {interp_err:.2e}",
          flush=True)
    for shape in RUN_SHAPES:
        for variant in RUN_VARIANTS:
            for src in RUN_NORMALS:
                label = cfg["name"] + NORMALS_SUFFIX[src]
                row = {"shape": shape, "variant": variant, "config": label, "normals": src,
                       "lengthscale": cfg["lengthscale"], "truncation_n": cfg["n"],
                       "grid_res": cfg["grid"], "kfv_interp_err": interp_err}
                try:
                    points = load_points(variant, shape)
                    gt = o3d.io.read_triangle_mesh(
                        str(DATA_DIR / variant / "03_meshes" / f"{shape}.ply"))
                    row["n_points"] = len(points)
                    print(f"[{label} / {shape} / {variant}] {len(points)} points ...",
                          flush=True)

                    normals = normals_gt(points, gt) if src == "gt" else None
                    mesh, timing = reconstruct(points, kern, sdd_params_for(len(points)),
                                               normals=normals)
                    chamfer, nc, fscore = mesh_metrics(mesh, gt)

                    o3d.io.write_triangle_mesh(
                        str(OUT_DIR / f"{result_stem(shape, variant, label)}.ply"), mesh)
                    row.update(chamfer=chamfer, normal_consistency=nc, fscore=fscore,
                               n_faces=len(mesh.triangles),
                               **{f"t_{k}": v for k, v in timing.items()})
                    print(f"    CD={chamfer:.5f}  NC={nc:.4f}  F1={fscore:.4f}  "
                          f"time={sum(timing.values()):.1f}s", flush=True)
                except Exception as e:            # keep going; record the failure
                    row["error"] = f"{type(e).__name__}: {e}"
                    print(f"    FAILED: {row['error']}", flush=True)
                results.append(row)

In [ ]:
cols = ["shape", "variant", "config", "normals", "lengthscale", "truncation_n", "grid_res",
        "kfv_interp_err", "n_points", "chamfer", "normal_consistency", "fscore", "n_faces",
        "t_normals", "t_solve+eval", "t_marching_cubes", "error"]
print(f"{'variant':<20}{'config':<28}{'N':>7}{'CD':>10}{'NC':>8}{'F1':>8}{'time[s]':>9}")
for r in results:
    head = f"{r['variant']:<20}{r['config']:<28}{r.get('n_points', ''):>7}"
    if "error" in r:
        print(f"{head}  {r['error']}")
        continue
    total = r["t_normals"] + r["t_solve+eval"] + r["t_marching_cubes"]
    print(f"{head}{r['chamfer']:>10.5f}{r['normal_consistency']:>8.4f}"
          f"{r['fscore']:>8.4f}{total:>9.1f}")

with open(OUT_DIR / "results.csv", "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)
print(f"saved -> {OUT_DIR}")

In [ ]:
# ---- Uncertainty pass: pathwise samples at the saved mean-mesh vertices ----
# For each saved mean mesh: draw S posterior samples of f (pathwise conditioning
# + SDD) at the data points and the mesh vertices.
#   sigma_f : std over samples of (f_s(v) - iso_s)           (level-set units)
#   sigma_d : sigma_f / |d mu / d n|  -> how far the surface may move along its
#             normal, in dataset units (same units as Chamfer / error to GT)
# Calibration check: Spearman rank correlation between sigma_d and the actual
# per-vertex distance to the GT mesh. Positive = high sigma where error is high.
from scipy.stats import spearmanr


def vertex_uncertainty(points, mesh, kern, n_samples, sdd_params, key, pad=1.1):
    """
    points: raw (n, 3) cloud, mesh: mean mesh from `reconstruct` (dataset coords)
    kern: the same `make_kernels` dict the mesh was built with
    returns: sigma_f (nv,), sigma_d (nv,), timings dict
    """
    t = {}
    center, scale = normalization_transform(points)
    x = (points - center) / scale * np.pi
    n = normals_o3d(x)                             # same normals as `reconstruct`
    mesh.compute_vertex_normals()                  # uniform scale -> same directions
    V = (np.asarray(mesh.vertices) - center) / scale * np.pi
    N = np.asarray(mesh.vertex_normals)
    x_data, v_data = jnp.asarray(x), jnp.asarray(n)
    n_freq = kern["v_evecs"].shape[0]

    t0 = time.perf_counter()
    xi = random.normal(key, shape=(n_samples, 3, n_freq, 2))
    gamma = f_gamma_from_v_xi(xi, kern["v_evecs"], kern["v_evals"])
    samples = sample_pathwise_conditioning(
        jnp.concatenate([x_data, jnp.asarray(V)]), x_data, v_data,
        kern["k_v"], kern["k_fv"], sigma, xi, gamma,
        kern["v_evecs"], kern["v_evals"], kern["v_evecs"], kern["f_evals"],
        # prior temporaries are (bs, S, 3, n_freq) for v and (bs, S, n_freq) for f
        bs_v=freq_batch_size(n_freq, n_samples * 3 * 3),
        bs_kfv=query_batch_size(len(x)),
        bs_f=freq_batch_size(n_freq, n_samples * 3),
        sdd_params=sdd_params,
    )
    samples = np.array(samples.block_until_ready())          # (S, nd + nv)
    t["samples"] = time.perf_counter() - t0

    s_data, s_vert = samples[:, : len(x)], samples[:, len(x):]
    iso_s = s_data.mean(1, keepdims=True)          # each sample has its own level
    sigma_f = (s_vert - iso_s).std(0, ddof=1)

    # |d mu / d n| at the vertices by central difference of the mean (one grid step)
    t0 = time.perf_counter()
    h = np.pi * pad / (kern["cfg"]["grid"] - 1)
    mu = np.array(compute_mean(
        jnp.asarray(np.concatenate([V + h * N, V - h * N])), x_data, v_data,
        kern["k_v"], kern["k_fv"], sigma, bs=query_batch_size(len(x)),
        sdd_params=sdd_params,
    ))
    grad_n = np.abs(mu[: len(V)] - mu[len(V):]) / (2 * h)
    t["grad"] = time.perf_counter() - t0

    grad_n = np.maximum(grad_n, 1e-3 * np.median(grad_n))   # avoid /0 at flat spots
    sigma_d = sigma_f / grad_n / np.pi * scale     # normalized -> dataset units
    return sigma_f, sigma_d, t


def vertex_error(mesh, gt):
    """unsigned distance from each mesh vertex to the GT surface"""
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(gt))
    q = o3d.core.Tensor(np.asarray(mesh.vertices), dtype=o3d.core.float32)
    return scene.compute_distance(q).numpy()

In [ ]:
# Optional and slow (RTX 4070: ~1 min per variant at n=50, ~4 min at n=70, dense
# ~2x; cost grows with (#points + #vertices) and with n^3). The mean meshes and
# metrics above don't need it; without it the demo shows "-" for sigma.
RUN_UNCERTAINTY = False
N_UNC_SAMPLES = 8          # each sample = one SDD solve; std from fewer is very noisy
UNC_CONFIGS = RUN_CONFIGS if RUN_UNCERTAINTY else []   # e.g. RUN_CONFIGS[:1]
UNC_VARIANTS = RUN_VARIANTS

unc_results = []
for cfg in UNC_CONFIGS:
    kern = make_kernels(cfg)
    for shape in RUN_SHAPES:
        for variant in UNC_VARIANTS:
            row = {"shape": shape, "variant": variant, "config": cfg["name"],
                   "n_samples": N_UNC_SAMPLES}
            stem = result_stem(shape, variant, cfg["name"])   # estimated-normals meshes
            try:
                if not (OUT_DIR / f"{stem}.ply").exists():
                    raise FileNotFoundError(f"run the mean batch first: {stem}.ply")
                points = load_points(variant, shape)
                mesh = o3d.io.read_triangle_mesh(str(OUT_DIR / f"{stem}.ply"))
                gt = o3d.io.read_triangle_mesh(
                    str(DATA_DIR / variant / "03_meshes" / f"{shape}.ply"))
                print(f"[{cfg['name']} / {shape} / {variant}] {len(points)} pts, "
                      f"{len(mesh.vertices)} verts ...", flush=True)

                sigma_f, sigma_d, timing = vertex_uncertainty(
                    points, mesh, kern, N_UNC_SAMPLES, sdd_params_for(len(points)),
                    random.key(1))
                err = vertex_error(mesh, gt)
                rho = spearmanr(sigma_d, err).statistic

                np.savez(OUT_DIR / f"{stem}_unc.npz",
                         sigma_f=sigma_f, sigma_d=sigma_d, err=err)
                row.update(sigma_d_mean=sigma_d.mean(), err_mean=err.mean(),
                           spearman=rho, **{f"t_{k}": v for k, v in timing.items()})
                print(f"    mean sigma_d={sigma_d.mean():.5f}  mean err={err.mean():.5f}  "
                      f"spearman={rho:.3f}  time={sum(timing.values()):.1f}s", flush=True)
            except Exception as e:
                row["error"] = f"{type(e).__name__}: {e}"
                print(f"    FAILED: {row['error']}", flush=True)
            unc_results.append(row)

unc_cols = ["shape", "variant", "config", "n_samples", "sigma_d_mean", "err_mean",
            "spearman", "t_samples", "t_grad", "error"]
if unc_results:                       # skipped -> keep any previous uncertainty.csv
    with open(OUT_DIR / "uncertainty.csv", "w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=unc_cols, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(unc_results)

In [ ]:
# ---- Demo: load saved results only (no reconstruction) ----
# Self-contained: run just this cell (no JAX / geospsr / cells above needed).
import csv
import subprocess
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyprojroot import here

DEMO_SHAPE = "xyzrgb_statuette"
DEMO_CONFIGS = ("ls0.01_n50_r128", "ls0.007_n70_r128", "ls0.005_n100_r128")
DEMO_DATA_DIR = here("data/ppsurf")
DEMO_OUT_DIR = here("data/results/geospsr_configs")
DEMO_TIME_COLS = ("t_normals", "t_solve+eval", "t_marching_cubes")
DEMO_VARIANTS = ("famous_noisefree", "famous_original", "famous_extra_noisy",
                 "famous_sparse", "famous_dense")
# every figure is also saved as a page here that fills the browser window and
# resizes with it (the inline plot has a fixed height)
DEMO_HTML_DIR = DEMO_OUT_DIR / "figures"
DEMO_GTN = "_gtnormals"        # label suffix of the GT-normals runs (NORMALS_SUFFIX)
DEMO_OPEN_BROWSER = False      # True: also open each page in the Windows browser


def _read_rows(path):
    if not path.exists():
        print(f"(missing {path.name})")
        return {}
    with open(path, newline="") as fh:
        return {(r["shape"], r["variant"], r["config"]): r
                for r in csv.DictReader(fh)}


demo_rows = _read_rows(DEMO_OUT_DIR / "results.csv")
demo_unc_rows = _read_rows(DEMO_OUT_DIR / "uncertainty.csv")


def _short(variant):
    return variant.removeprefix("famous_")


def _ok(r):
    return r is not None and not r["error"]


def demo_table():
    print("sigma/err: mean over vertices (dataset units), rho: Spearman(sigma, err)")
    print(f"{'variant':<13}{'config':<28}{'N':>7}{'CD':>9}{'NC':>7}{'F1':>7}"
          f"{'sigma':>9}{'err':>9}{'rho':>7}{'time[s]':>9}")
    for key, r in sorted(demo_rows.items(), key=lambda kv: (kv[0][1], kv[0][2])):
        head = f"{_short(r['variant']):<13}{r['config']:<28}{r['n_points']:>7}"
        if not _ok(r):
            print(f"{head}  {r['error']}")
            continue
        u = demo_unc_rows.get(key)
        unc = (f"{float(u['sigma_d_mean']):>9.5f}{float(u['err_mean']):>9.5f}"
               f"{float(u['spearman']):>7.3f}") if _ok(u) else f"{'-':>9}{'-':>9}{'-':>7}"
        total = sum(float(r[c]) for c in DEMO_TIME_COLS)
        print(f"{head}{float(r['chamfer']):>9.5f}{float(r['normal_consistency']):>7.3f}"
              f"{float(r['fscore']):>7.3f}{unc}{total:>9.1f}")


def _title(main, sub=""):
    """bold short main line + small second line, so titles stay narrower than a column"""
    return f"<b>{main}</b>" + (f"<br><span style='font-size:10px'>{sub}</span>" if sub else "")


def _metric_sub(shape, variant, config):
    r = demo_rows.get((shape, variant, config))
    return f"CD {float(r['chamfer']):.4f} · F1 {float(r['fscore']):.2f}" if _ok(r) else ""


def _mesh_trace(mesh, max_faces, color="lightgray", values=None):
    """values: per-vertex scalars -> colored by the shared coloraxis; else flat `color`"""
    if values is None and len(mesh.triangles) > max_faces:   # keep the browser responsive
        mesh = mesh.simplify_quadric_decimation(max_faces)    # (would break per-vertex values)
    v, f = np.asarray(mesh.vertices), np.asarray(mesh.triangles)
    kw = dict(color=color) if values is None else dict(
        intensity=values, intensitymode="vertex", coloraxis="coloraxis")
    return go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2],
                     i=f[:, 0], j=f[:, 1], k=f[:, 2], hoverinfo="skip",
                     lighting=dict(ambient=0.35, diffuse=0.9, specular=0.15,
                                   roughness=0.6, fresnel=0.1), **kw)


def _points_trace(points, max_points=30_000):
    if len(points) > max_points:
        points = points[np.random.default_rng(0).choice(len(points), max_points, replace=False)]
    return go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="markers",
                        marker=dict(size=1, color="gray"), hoverinfo="skip")


_gt_cache = {}


def _gt_mesh(shape, max_faces, variant="famous_noisefree"):
    """GT is 1M triangles (same in every variant): decimate once per (shape, size)"""
    key = (shape, max_faces)
    if key not in _gt_cache:
        gt = o3d.io.read_triangle_mesh(str(DEMO_DATA_DIR / variant / "03_meshes" / f"{shape}.ply"))
        _gt_cache[key] = gt.simplify_quadric_decimation(max_faces)
    return _gt_cache[key]


def _gt_trace(shape, max_faces):
    return _mesh_trace(_gt_mesh(shape, max_faces), max_faces, "lightgray")


def _camera(shape):
    """
    The shapes are rotated arbitrarily in the dataset, so no axis view works.
    Use the GT's principal axes: up = longest axis, look along the 2nd one,
    slightly from above. Every panel gets the same camera.
    """
    v = np.asarray(_gt_mesh(shape, 50_000).vertices)
    _, E = np.linalg.eigh(np.cov((v - v.mean(0)).T))   # ascending variance
    up, side = E[:, 2], E[:, 1]
    eye = 2.3 * side + 0.5 * up
    return dict(eye=dict(zip("xyz", eye)), up=dict(zip("xyz", up)))


def _stem(shape, variant, config):
    return DEMO_OUT_DIR / f"{shape}_{variant}_{config}"


def _rec_mesh(shape, variant, config):
    path = DEMO_OUT_DIR / f"{_stem(shape, variant, config).name}.ply"
    if not path.exists():
        print(f"missing: {path.name}")
        return None
    return o3d.io.read_triangle_mesh(str(path))


def _unc(shape, variant, config):
    """{'sigma_d', 'sigma_f', 'err'} per vertex of the saved mean mesh, or None"""
    path = DEMO_OUT_DIR / f"{_stem(shape, variant, config).name}_unc.npz"
    return dict(np.load(path)) if path.exists() else None


def _save_page(fig, name):
    """fig as DEMO_HTML_DIR/<name>.html: fills the browser window and follows its size"""
    DEMO_HTML_DIR.mkdir(parents=True, exist_ok=True)
    path = DEMO_HTML_DIR / f"{name}.html"
    page = go.Figure(fig)                  # copy, so the inline figure keeps its height
    page.layout.height = None              # size comes from the window instead
    # body has 8px margins -> calc() avoids a scrollbar; plotly.min.js is written once
    page.write_html(path, include_plotlyjs="directory", config={"responsive": True},
                    default_width="100%", default_height="calc(100vh - 16px)")
    try:                                   # WSL: Windows path to paste into Explorer / browser
        shown = subprocess.run(["wslpath", "-w", str(path)], capture_output=True,
                               text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        shown = str(path)
    print(f"page: {shown}")
    if DEMO_OPEN_BROWSER:
        subprocess.run(["explorer.exe", shown])


def _demo_fig(titles, traces, camera, height=480, colorbar=None, name=None):
    """
    camera: from `_camera`, applied to every panel
    colorbar: (first_col, last_col, title, cmax) -> ONE horizontal bar under those
    columns for the shared coloraxis (separate per-trace bars would stack on the plots)
    name: also save it as a resizable browser page (`_save_page`)
    """
    n = len(titles)
    fig = make_subplots(rows=1, cols=n, subplot_titles=titles,
                        specs=[[{"type": "scene"}] * n], horizontal_spacing=0.03)
    for col, tr in enumerate(traces, start=1):
        fig.add_trace(tr, row=1, col=col)
    fig.update_annotations(font_size=13)
    fig.update_scenes(aspectmode="data", xaxis_visible=False, yaxis_visible=False,
                      zaxis_visible=False, camera=camera)
    extra = {}
    if colorbar:
        c0, c1, cb_title, cmax = colorbar
        x0 = fig.layout["scene" if c0 == 1 else f"scene{c0}"].domain.x[0]
        x1 = fig.layout["scene" if c1 == 1 else f"scene{c1}"].domain.x[1]
        extra["coloraxis"] = dict(
            colorscale="Viridis", cmin=0, cmax=cmax,
            colorbar=dict(orientation="h", x=(x0 + x1) / 2, xanchor="center",
                          y=-0.02, yanchor="top", len=0.6 * (x1 - x0), thickness=10,
                          title=dict(text=cb_title, side="top", font_size=11),
                          tickfont_size=10, nticks=5, tickformat=".3f"))
    fig.update_layout(height=height, showlegend=False,
                      margin=dict(l=10, r=10, t=70, b=80 if colorbar else 10), **extra)
    fig.show()
    if name:
        _save_page(fig, name)


def demo_case(variant, shape=DEMO_SHAPE, configs=DEMO_CONFIGS, max_faces=100_000):
    """input points | GT mesh | GeoSPSR mean mesh for each config"""
    pcd = o3d.io.read_point_cloud(str(DEMO_DATA_DIR / variant / "04_pts_vis" / f"{shape}.xyz.ply"))
    titles = [_title("input", f"{_short(variant)} · {len(pcd.points)} pts"), _title("GT")]
    traces = [_points_trace(np.asarray(pcd.points)), _gt_trace(shape, max_faces)]
    for c in configs:
        rec = _rec_mesh(shape, variant, c)
        if rec is not None:
            titles.append(_title(c, _metric_sub(shape, variant, c)))
            traces.append(_mesh_trace(rec, max_faces, "lightskyblue"))
    _demo_fig(titles, traces, _camera(shape), name=f"case_{shape}_{_short(variant)}")


def demo_variants(config, shape=DEMO_SHAPE, variants=DEMO_VARIANTS, max_faces=50_000):
    """GT | GeoSPSR mean mesh of every variant, one config"""
    titles = [_title("GT", config)]            # run label: est. vs GT-normals figures
    traces = [_gt_trace(shape, max_faces)]
    for v in variants:
        rec = _rec_mesh(shape, v, config)
        if rec is not None:
            titles.append(_title(_short(v), _metric_sub(shape, v, config)))
            traces.append(_mesh_trace(rec, max_faces, "lightskyblue"))
    _demo_fig(titles, traces, _camera(shape), height=420, name=f"variants_{shape}_{config}")


def demo_uncertainty(variant, config, shape=DEMO_SHAPE, max_faces=100_000):
    """GT | mean mesh colored by predicted sigma | same mesh colored by actual error"""
    rec, unc = _rec_mesh(shape, variant, config), _unc(shape, variant, config)
    if rec is None or unc is None:
        print(f"(no uncertainty for {_short(variant)} {config}: run the uncertainty cell)")
        return
    # one color range for both -> predicted sigma and actual error directly comparable
    cmax = float(np.percentile(np.concatenate([unc["sigma_d"], unc["err"]]), 95))
    u = demo_unc_rows.get((shape, variant, config))
    rho = f" · Spearman w/ err {float(u['spearman']):.2f}" if _ok(u) else ""
    _demo_fig(
        [_title("GT"),
         _title("uncertainty σ", "predicted, no GT" + rho),
         _title("error to GT", _metric_sub(shape, variant, config))],
        [_gt_trace(shape, max_faces),
         _mesh_trace(rec, max_faces, values=unc["sigma_d"]),
         _mesh_trace(rec, max_faces, values=unc["err"])],
        _camera(shape),
        colorbar=(2, 3, f"distance (dataset units) · {_short(variant)} · {config}", cmax),
        name=f"uncertainty_{shape}_{_short(variant)}_{config}",
    )

In [ ]:
demo_table()
demo_case("famous_noisefree")          # input | GT | every config
for c in DEMO_CONFIGS:
    demo_variants(c)                   # GT | every variant, estimated normals
    demo_variants(c + DEMO_GTN)        # same with GT normals (no normal-estimation error)
demo_uncertainty("famous_noisefree", DEMO_CONFIGS[0])